**Clean Raw Data**

In [ ]:
import pandas as pd
import os

files = [
    'DrDoS_DNS.csv',
    'DrDoS_LDAP.csv',
    'DrDoS_MSSQL.csv',
    'DrDoS_NetBIOS.csv',
    'DrDoS_NTP.csv',
    'DrDoS_SNMP.csv',
    'DrDoS_SSDP.csv',
    'DrDoS_UDP.csv',
    'Syn.csv',
    'TFTP.csv',
    'UDPLag.csv',
]

results = []

for file in files:
    df = pd.read_csv(file, usecols=[' Label'])
    counts = df[' Label'].value_counts()
    results.append({
        'File Name': file,
        'Attack Count': counts.get('DrDoS_DNS', counts[counts.index != 'BENIGN'].sum()),
        'Benign Count': counts.get('BENIGN', 0),
    })

pd.DataFrame(results)

Data cleaning

In [ ]:
import pandas as pd
import numpy as np
import os

# raw CICDDoS2019 files in the same folder as this notebook
raw_files = [
    'DrDoS_DNS.csv',
    'DrDoS_LDAP.csv',
    'DrDoS_MSSQL.csv',
    'DrDoS_NetBIOS.csv',
    'DrDoS_NTP.csv',
    'DrDoS_SNMP.csv',
    'DrDoS_SSDP.csv',
    'DrDoS_UDP.csv',
    'Syn.csv',
    'TFTP.csv',
    'UDPLag.csv'
]

def clean_file(input_file, output_file):
    print('Cleaning:', input_file)

    df = pd.read_csv(input_file, low_memory=False)

    # strip column names
    df.columns = df.columns.str.strip()

    # convert Label to 0/1 if exists
    if 'Label' in df.columns:
        df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

    # keep numeric columns + Label
    keep = []
    for col in df.columns:
        if col == 'Label':
            keep.append(col)
        else:
            if pd.api.types.is_numeric_dtype(df[col]):
                keep.append(col)

    df = df[keep]

    # replace inf and NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)

    # drop duplicates
    df = df.drop_duplicates()

    # save cleaned file
    df.to_csv(output_file, index=False)
    print('Saved:', output_file)
    print()

# run cleaning for each file
for f in raw_files:
    clean_file(f, 'clean_' + f)

print('All files cleaned.')


In [ ]:
import pandas as pd
import os
from sklearn.utils import resample

clean_files = [
    'clean_DrDoS_DNS.csv',
    'clean_DrDoS_LDAP.csv',
    'clean_DrDoS_MSSQL.csv',
    'clean_DrDoS_NetBIOS.csv',
    'clean_DrDoS_NTP.csv',
    'clean_DrDoS_SNMP.csv',
    'clean_DrDoS_SSDP.csv',
    'clean_DrDoS_UDP.csv',
    'clean_Syn.csv',
    'clean_TFTP.csv',
    'clean_UDPLag.csv'
]

def balance_file(input_file, output_file):
    print('Balancing:', input_file)

    df = pd.read_csv(input_file)

    if 'Label' not in df.columns:
        print('No Label column, skipping:', input_file)
        return

    df0 = df[df['Label'] == 0]
    df1 = df[df['Label'] == 1]

    if len(df0) == 0 or len(df1) == 0:
        print('Only one class present, skipping:', input_file)
        return

    min_size = min(len(df0), len(df1))

    df0_bal = resample(df0, replace=False, n_samples=min_size, random_state=42)
    df1_bal = resample(df1, replace=False, n_samples=min_size, random_state=42)

    df_balanced = pd.concat([df0_bal, df1_bal]).sample(frac=1, random_state=42)

    df_balanced.to_csv(output_file, index=False)
    print('Saved:', output_file)
    print()

for f in clean_files:
    balance_file(f, 'balanced_' + f)

print('All files balanced.')


Data cleaning and balancing

In [ ]:
import pandas as pd
import os

balanced_files = [
    'balanced_clean_DrDoS_DNS.csv',
    'balanced_clean_DrDoS_LDAP.csv',
    'balanced_clean_DrDoS_MSSQL.csv',
    'balanced_clean_DrDoS_NetBIOS.csv',
    'balanced_clean_DrDoS_NTP.csv',
    'balanced_clean_DrDoS_SNMP.csv',
    'balanced_clean_DrDoS_SSDP.csv',
    'balanced_clean_DrDoS_UDP.csv',
    'balanced_clean_Syn.csv',
    'balanced_clean_TFTP.csv',
    'balanced_clean_UDPLag.csv'
]

all_data = []

for f in balanced_files:
    if os.path.exists(f):
        print('Loading:', f)
        df = pd.read_csv(f)
        df['Source'] = f  # optional: track origin
        all_data.append(df)
    else:
        print('Missing file:', f)

combined = pd.concat(all_data, ignore_index=True)

combined = combined.drop_duplicates()

combined = combined.sample(frac=1, random_state=42)

combined.to_csv('combined_balanced.csv', index=False)

print('Saved combined_balanced.csv')
print('Shape:', combined.shape)


In [ ]:
df = pd.read_csv('combined_balanced.csv')
print(df['Label'].value_counts())

Feature selection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

#Load and clean dataset
df = pd.read_csv('combined_balanced.csv')

if 'Source' in df.columns:
    df = df.drop('Source', axis=1)

X = df.drop('Label', axis=1)
y = df['Label']

#train Random Forest to get feature importance
rf_importance = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_importance.fit(X, y)

importance = rf_importance.feature_importances_

feat_table = pd.DataFrame({
    'feature': X.columns,
    'importance': importance
}).sort_values(by='importance', ascending=False)

feat_table.to_csv('selected_features.csv', index=False)

top20 = feat_table.head(20)['feature'].tolist()

# ablation test with different K values
k_list = [3, 5, 8, 10, 13, 15, 18, 20]

results = []

for k in k_list:
    selected = top20[:k]
    X_sel = df[selected]

    X_train, X_test, y_train, y_test = train_test_split(
        X_sel, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Train model and evaluate
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        class_weight='balanced'
    )
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)

    results.append([k, acc])

results_df = pd.DataFrame(results, columns=['K', 'Accuracy'])
results_df.to_csv('ablation_results.csv', index=False)

plt.figure(figsize=(10, 6))
plt.plot(results_df['K'], results_df['Accuracy'], marker='o')
plt.xlabel('Number of Features (K)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Features')
plt.xticks(k_list)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('feature_ablation_accuracy.png', dpi=300)
plt.show()
for i, f in enumerate(top20[:15], start=1):
    print(i, f)




Centralized model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# load top 20 and take top 15
selected = pd.read_csv('selected_features.csv')
top15 = selected['feature'].tolist()[:15]

# load dataset
df = pd.read_csv('combined_balanced.csv')

X = df[top15]
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=2000
    ),
    'SVM': SVC(
        kernel='rbf',
        probability=True
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss'
    )
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append([name, acc, prec, rec, f1])

results_df = pd.DataFrame(
    results,
    columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score']
)

print(results_df)


**Create 3 Tenants (Non‑IID)**

In [ ]:
import pandas as pd

# Load clean attack files
files = {
    "DNS": "clean_DrDoS_DNS.csv",
    "LDAP": "clean_DrDoS_LDAP.csv",
    "MSSQL": "clean_DrDoS_MSSQL.csv",
    "NetBIOS": "clean_DrDoS_NetBIOS.csv",
    "NTP": "clean_DrDoS_NTP.csv",
    "SNMP": "clean_DrDoS_SNMP.csv",
    "SSDP": "clean_DrDoS_SSDP.csv",
    "UDP": "clean_DrDoS_UDP.csv",
    "UDPLag": "clean_UDPLag.csv"
}

dfs = []
for attack, path in files.items():
    df = pd.read_csv(path)
    df["Attack_Type"] = attack
    df.loc[df["Label"] == 0, "Attack_Type"] = "BENIGN"
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True).drop_duplicates()

# Load top 15 features
selected = pd.read_csv("selected_features.csv")
top15 = selected["feature"].tolist()[:15]

# Tenant definitions
tenant_attacks = {
    "tenant_1_mixed.csv": ["BENIGN", "DNS", "LDAP"],
    "tenant_2_mixed.csv": ["BENIGN", "MSSQL", "NetBIOS", "NTP"],
    "tenant_3_mixed.csv": ["BENIGN", "UDP", "SSDP", "SNMP", "UDPLag"]
}

# Tenant size multipliers
tenant_sizes = {
    "tenant_1_mixed.csv": 0.20,   # small
    "tenant_2_mixed.csv": 0.50,   # medium
    "tenant_3_mixed.csv": 1.00    # large
}

# Build tenants
for name, attacks in tenant_attacks.items():
    df_t = full[full["Attack_Type"].isin(attacks)]

    # Apply size multiplier
    frac = tenant_sizes[name]
    df_t = df_t.sample(frac=frac, random_state=42)

    # Balance AFTER sampling
    benign = df_t[df_t["Label"] == 0]
    attack = df_t[df_t["Label"] == 1]
    min_n = min(len(benign), len(attack))

    balanced = pd.concat([
        benign.sample(n=min_n, random_state=42),
        attack.sample(n=min_n, random_state=42)
    ]).sample(frac=1, random_state=42)

    # Keep Attack_Type for verification
    balanced.to_csv(name, index=False)
    print(name, balanced.shape)


Baseline  model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Load top 15 features
selected = pd.read_csv('selected_features.csv')
top15 = selected['feature'].tolist()[:15]

# Tenant datasets
tenant_files = {
    'Tenant 1 (Small)': 'tenant_1_mixed.csv',
    'Tenant 2 (Medium)': 'tenant_2_mixed.csv',
    'Tenant 3 (Large)': 'tenant_3_mixed.csv'
}

all_results = {}

for tenant, file in tenant_files.items():
    print(f'\nTraining models for: {tenant}')

    df = pd.read_csv(file)

    X = df[top15]
    y = df['Label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    models = {
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=2000),
        'SVM (RBF)': SVC(kernel='rbf', probability=True),
        'XGBoost': XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss'
        )
    }

    results = []

    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append([name, acc, prec, rec, f1])

    results_df = pd.DataFrame(
        results,
        columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score']
    )

    all_results[tenant] = results_df
    print(results_df)


Ablation study

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load tenant dataset
df = pd.read_csv("tenant_3_mixed.csv")

# Top 3 features
F1 = "Fwd Packet Length Min"
F2 = "Min Packet Length"
F3 = "Fwd Packet Length Mean"

feature_sets = {
    "3_features": [F1, F2, F3],
    "F1_F2": [F1, F2],
    "F1_F3": [F1, F3],
    "F2_F3": [F2, F3]
}

results = []

for name, feats in feature_sets.items():
    X = df[feats]
    y = df["Label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Features": feats,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

pd.DataFrame(results)


In [ ]:
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('tenant_3_mixed.csv')

selected = pd.read_csv('selected_features.csv')
top15 = selected['feature'].tolist()[:15]

X = df[top15]
y = df['Label']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

if isinstance(shap_values, list):
    vals = shap_values[1]
else:
    vals = shap_values[:, :, 1]

shap.summary_plot(vals, X, plot_type='bar', title='SHAP - Tenant 3')

INDIvidual tenants features

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('tenant_3_mixed.csv')
X = df.drop(columns=['Label', 'Attack_Type'], errors='ignore').select_dtypes(include='number')
y = df['Label']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print(importance.head(15).to_string())

Confusion Matrix

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Load features
selected = pd.read_csv('selected_features.csv')
top15 = selected['feature'].tolist()[:15]


def plot_cm(cm, title):
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'])
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()


# Load centralized dataset
df = pd.read_csv('combined_balanced.csv')
X = df[top15]
y = df['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'SVM': SVC(kernel='rbf', probability=True),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8, eval_metric='logloss'),
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    print(f'{name}:\n{cm}\n')
    plot_cm(cm, f'Centralized - {name}')

Ablation Study

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv('combined_balanced.csv')

F1 = 'Fwd Packet Length Min'
F2 = 'Min Packet Length'
F3 = 'Fwd Packet Length Mean'
F4 = 'Destination Port'

feature_sets = {
    'None (all 4)':      [F1, F2, F3, F4],
    'Fwd Packet Length Min':  [F2, F3, F4],
    'Min Packet Length':      [F1, F3, F4],
    'Fwd Packet Length Mean': [F1, F2, F4],
    'Destination Port':       [F1, F2, F3],
}

results = []

for removed, feats in feature_sets.items():
    X = df[feats]
    y = df['Label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        'Removed Feature': removed,
        'Accuracy': round(accuracy_score(y_test, y_pred), 6)
    })

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))